Este script sirve para aplicar data augmentation al conjunto de datos original de xview-recognition. 
Hay dos opciones: balancear las clases generando nuevos ejemplos hasta llegar al número de la clase con más ejemplos 
o elegir este número al que deben llegar todas arbitrariamente.

Las imágenes están en una carpeta 'xview_train' que tiene 1 subcarpeta por cada categoría de objetos, donde se encuentran todas las imágenes de objetos de ese tipo.
Cada imagen tiene un nombre_imagen único referenciado desde un JSON. Este se usa a modo de índice, y esencialmente contiene pares (nombre_imagen, categoría). 
El objetivo es introducir nuevas imágenes en su directorio correspondiente y actualizar el JSON para indexarlas.

Proceso:
Se lee el JSON para obtener el número de imágenes de cada tipo. 
Calcular cuántas imágenes hay que generar de cadat categoría.
Para cada las categoría, hasta llegar al número necesario:
    Generar una imágenes una transformacion aleatoria con keras de una imagen aleatoria de esa categoría.
    Guardarla en la carpeta de la categoría correcta y refereciarla en el JSON para que la indexe.

In [26]:
import uuid
import numpy as np

class GenericObject:
    """
    Generic object data.
    """
    def __init__(self):
        self.id = uuid.uuid4()
        self.bb = (-1, -1, -1, -1)
        self.category= -1
        self.score = -1

class GenericImage:
    """
    Generic image data.
    """
    def __init__(self, filename):
        self.filename = filename
        self.tile = np.array([-1, -1, -1, -1])  # (pt_x, pt_y, pt_x+width, pt_y+height)
        self.objects = list([]) # Realmente solo hay 1 objeto por cada imagen

    def add_object(self, obj: GenericObject):
        self.objects.append(obj)

In [27]:
categories = {0: 'Cargo plane', 1: 'Helicopter', 2: 'Small car', 3: 'Bus', 4: 'Truck', 5: 'Motorboat', 6: 'Fishing vessel', 7: 'Dump truck', 8: 'Excavator', 9: 'Building', 10: 'Storage tank', 11: 'Shipping container'}

In [28]:
import json
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

import shutil
import os

def copy_dataset_to_output(input_directory, output_directory):
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)

    # Ahora copia todos los archivos del directorio de entrada al de salida
    for filename in os.listdir(input_directory):
        input_file = os.path.join(input_directory, filename)
        output_file = os.path.join(output_directory, filename)

        # Si es un archivo, lo copiamos
        if os.path.isfile(input_file):
            shutil.copy(input_file, output_file)
    
    
    # Para directorios (y su contenido) dentro del directorio de entrada, copiarlos de forma recursiva
    for dirname in os.listdir(input_directory):
        input_dir_path = os.path.join(input_directory, dirname)
        output_dir_path = os.path.join(output_directory, dirname)

        # Si es un directorio, copia de forma recursiva
        if os.path.isdir(input_dir_path):
            shutil.copytree(input_dir_path, output_dir_path)
    
# Load JSON and create a dictionary with category: list of images
def import_database(json_file):
    # Load database JSON.
    with open(json_file) as ifs:
        json_data = json.load(ifs)
    ifs.close()
    
    categories_images = {category: [] for category in categories.values()}

    # Crear un GenericImage por cada archivo con su GenericObject correspondiente categorizado dentro
    for json_img, json_ann in zip(json_data['images'].values(), json_data['annotations'].values()):
        image = GenericImage(json_img['filename'])
        image.tile = np.array([0, 0, json_img['width'], json_img['height']])
        obj = GenericObject()
        obj.bb = (int(json_ann['bbox'][0]), int(json_ann['bbox'][1]), int(json_ann['bbox'][2]), int(json_ann['bbox'][3]))
        obj.category = json_ann['category_id']
        
        image.add_object(obj)
        categories_images[obj.category].append(image)

    return categories_images

# Actualiza el json con todas las nuevas imágenes (empieza con el id máximo + 1 del dataset original)
def update_json(json_file, new_images, start_id):
    
    # Load database JSON.
    with open(json_file) as ifs:
        json_data = json.load(ifs)
    ifs.close()
    
    id = start_id
    # Añade nuevas entradas a las listas
    for image in new_images:
        filename = image.filename
        category = image.objects[0].category
        image_json_entry = {
                     "image_id": filename, 
                    "filename": f"xview_train/{category}/{filename}", 
                    "width": 224, 
                    "height": 224}
        
        image_category_json_entry = {
                    "image_id": filename, 
                      "category_id": category, 
                      "bbox": [0, 0, 224, 224]}    

        json_data['images'][str(id)] = image_json_entry
        json_data['annotations'][str(id)] = image_category_json_entry
        
        id = id + 1

    # Escribe el archivo JSON actualizado
    with open(json_file, 'w') as file:
        json.dump(json_data, file)

In [29]:
import warnings
import rasterio
import numpy as np
import os

# Devuelve la matriz 3D con los bits de una imagen dado el nombre del archivo
def load_geoimage(filename):
    filename = '/kaggle/working/xviewrecognitionaug/' + filename
    warnings.filterwarnings('ignore', category=rasterio.errors.NotGeoreferencedWarning)
    src_raster = rasterio.open(filename, 'r')
    # RasterIO to OpenCV (see inconsistencies between libjpeg and libjpeg-turbo)
    input_type = src_raster.profile['dtype']
    input_channels = src_raster.count # 3 canales (RGB)
    img = np.zeros((src_raster.height, src_raster.width, src_raster.count), dtype=input_type)
    for band in range(input_channels): # Rellenar toda la matriz 2D del canal k de la matriz
        img[:, :, band] = src_raster.read(band+1)
    return img

# Creates a new image object
def new_image(filename, category):
    image = GenericImage(filename)
    obj = GenericObject()
    obj.category = category
    image.add_object(obj)
    
    return image

# Generates and saves a variation of an input image of some category
def generate_image(datagen, image_bin, category, filename, i):
    
    category_dir = f"/kaggle/working/xviewrecognitionaug/xview_train/{category}"
    filename = filename.split('/')[-1]
    filename = f"aug_{filename}_{i}.tif" # Para saber cual era la original
    output_path = os.path.join(category_dir, filename)
    
    image_transformed = datagen.random_transform(image_bin)
    # rasterio espera un array de 3D en orden (bandas, filas, columnas)
    image_transformed = np.moveaxis(image_transformed, 2, 0)
    with rasterio.Env():
        profile = {
        'driver': 'GTiff',
        'height': image_transformed.shape[1],
        'width': image_transformed.shape[2],
        'count': image_transformed.shape[0],
        'dtype': 'uint8',  # Asegúrate de que tus datos estén en el tipo de datos correcto para 24 bits (8 bits por canal)
        'compress': 'lzw',
        'photometric': 'RGB'  # Esto es necesario si estás guardando imágenes RGB
    }
        with rasterio.open(output_path, 'w', **profile) as dst:
            dst.write(image_transformed)
    
    image = new_image(filename, category)
    return image

In [33]:
### import os
import numpy as np
import random
from tensorflow.keras.preprocessing.image import ImageDataGenerator

MAX_CATEGORY_AUG = 1 # Cuantas veces se va a aumentar en base a la clase más frecuente (1 para igualarla y != 1 aumentar el resto hasta ese porcentaje)

# Inicializa el generador de data augmentation con las transformaciones deseadas
datagen = ImageDataGenerator(
    rotation_range=180,          # Rotaciones completas (de 0 a 180 grados)
    width_shift_range=0.1,       # Traslaciones horizontales (10% del total)
    height_shift_range=0.1,      # Traslaciones verticales (10% del total)
    shear_range=0.05,            # Rango de deformación cizalla (5%)
    zoom_range=[0.9, 1.1],       # Zoom de 90% a 110%
    horizontal_flip=True,        # Activar para girar las imágenes horizontalmente
    vertical_flip=True,          # Activar para girar las imágenes verticalmente
    fill_mode='nearest',         # 'nearest' para que la estructura continúe
    brightness_range=[0.8, 1.2], # Rango de brillo (80% a 120% del original)
)

# Ruta al directorio de input (solo lectura)
input_directory = '../input/xviewrecognition/'

# Ruta al directorio de escritura
output_directory = '/kaggle/working/xviewrecognitionaug/'

if os.path.exists(output_directory):
    # Borrar el directorio y todos los archivos contenidos
    shutil.rmtree(output_directory)

copy_dataset_to_output(input_directory, output_directory)

print('Dataset copiado a output')

# Category: list images
categories_images = import_database(output_directory + 'xview_ann_train.json')

print(categories_images.keys())

max_category_instances = max(len(images) for images in categories_images.values())
target_instances_per_category = round(max_category_instances * MAX_CATEGORY_AUG)
total_instances = sum(len(images) for images in categories_images.values())

new_images = []

# Recorre cada categoría y sus imágenes
for category, images in categories_images.items():
    
    num_to_generate = target_instances_per_category - len(images)
    print(f'Category: {category}, num to generate: {num_to_generate}')
    for i in range(0,num_to_generate):
        random_image = random.choice(images)
        filename = random_image.filename
        image_bin = load_geoimage(filename)
        
        generated_image = generate_image(datagen, image_bin, category, filename, i)
        new_images.append(generated_image)
        
update_json(output_directory + 'xview_ann_train.json', new_images, total_instances)

print('Updated JSON')

dict_keys(['Cargo plane', 'Helicopter', 'Small car', 'Bus', 'Truck', 'Motorboat', 'Fishing vessel', 'Dump truck', 'Excavator', 'Building', 'Storage tank', 'Shipping container'])
Category: Cargo plane, num to generate: -166
Category: Helicopter, num to generate: 399
Category: Small car, num to generate: -3821
Category: Bus, num to generate: -1686
Category: Truck, num to generate: -2277
Category: Motorboat, num to generate: -600
Category: Fishing vessel, num to generate: -237
Category: Dump truck, num to generate: -767
Category: Excavator, num to generate: -320
Category: Building, num to generate: -4220
Category: Storage tank, num to generate: -1000
Category: Shipping container, num to generate: -1054
Updated JSON


In [31]:
import shutil

# Comprime el directorio /kaggle/working/xviewrecognitionaug
shutil.make_archive('/kaggle/working/xviewrecognitionaug', 'zip', '/kaggle/working/xviewrecognitionaug')

if os.path.exists(output_directory):
    # Borrar el directorio y todos los archivos contenidos
    shutil.rmtree(output_directory)